# SmolVLA Parol6 Gripper 测试
## 测试 Parol6 机械臂的夹爪控制

⚠️ **注意**: 此版本包含一些常见问题，请参考 `9.test_parol6_gripper_fixed.ipynb` 获取修复版本

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import time
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

## 1. 加载模型和Tokenizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 加载模型
model_name = "lerobot/smolvla_base"
policy = SmolVLAPolicy.from_pretrained(model_name)
policy = policy.to(device).eval()

# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Video-Instruct")
print("模型加载完成!")

## 2. 准备测试数据

### ❌ 问题 1: 缺少数据集加载和归一化统计

In [ ]:
# ❌ 问题: 没有加载数据集统计信息
# 应该从数据集中获取归一化参数

# 创建模拟图像数据
batch_size = 1
image_size = 256

# Parol6 模拟相机输入 (3个相机)
camera1_img = torch.rand(batch_size, 3, image_size, image_size).to(device)
camera2_img = torch.rand(batch_size, 3, image_size, image_size).to(device)
camera3_img = torch.rand(batch_size, 3, image_size, image_size).to(device)

## 3. 准备关节状态

### ❌ 问题 2: Gripper 维度处理不正确

In [ ]:
# Parol6: 6 DOF + 1 gripper = 7维动作空间
# ❌ 问题: 只考虑了6维，没有正确处理gripper

# 当前关节状态 (6个关节角度)
joint_positions = torch.tensor([
    [0.0, 45.0, 90.0, 0.0, 45.0, 0.0]  # 单位: 度
]).to(device)

print(f"关节状态维度: {joint_positions.shape}")
print(f"关节位置: {joint_positions}")

## 4. 准备语言指令

In [ ]:
# 任务描述
task_text = "Grasp the object with the gripper."

# Tokenize
text_tokens = tokenizer(task_text, return_tensors="pt")
lang_tokens = text_tokens['input_ids'].to(device)
lang_attention_mask = text_tokens['attention_mask'].to(device).bool()

print(f"任务: {task_text}")
print(f"Token数量: {lang_tokens.shape[1]}")

## 5. 构建输入批次

### ❌ 问题 3: 没有包含gripper状态

In [ ]:
# ❌ 问题: observation.state只有6维，缺少gripper状态
batch = {
    'observation.images.camera1': camera1_img,
    'observation.images.camera2': camera2_img,
    'observation.images.camera3': camera3_img,
    'observation.state': joint_positions,  # 应该是7维 (6关节 + 1gripper)
    'observation.language.tokens': lang_tokens,
    'observation.language.attention_mask': lang_attention_mask
}

print("\n输入批次准备完成:")
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")

## 6. 运行推理

### ❌ 问题 4: 没有反归一化

In [ ]:
# 推理
with torch.no_grad():
    start_time = time.time()
    output = policy.select_action(batch)
    inference_time = (time.time() - start_time) * 1000

print(f"\n推理完成!")
print(f"推理时间: {inference_time:.2f}ms")
print(f"输出形状: {output.shape}")

# ❌ 问题: 直接使用归一化的输出，没有反归一化
predicted_action = output[0].cpu().numpy()

print(f"\n预测动作:")
print(f"  维度: {predicted_action.shape}")
print(f"  值域: [{predicted_action.min():.4f}, {predicted_action.max():.4f}]")
print(f"  动作: {predicted_action[:6]}")

# ❌ 问题: 归一化的值（通常在[-1, 1]之间）被当作实际角度使用
if predicted_action.shape[0] >= 7:
    gripper_value = predicted_action[6]
    print(f"  Gripper: {gripper_value:.4f}")
else:
    print(f"  ⚠️  警告: 输出维度不包含gripper!")

## 7. 解析 Gripper 命令

### ❌ 问题 5: Gripper 阈值判断不正确

In [ ]:
# ❌ 问题: 使用归一化值判断gripper状态
# 归一化值通常在[-1, 1]范围内，不应该直接与0.5比较

if predicted_action.shape[0] >= 7:
    gripper_value = predicted_action[6]
    
    # ❌ 错误的阈值判断
    if gripper_value > 0.5:
        gripper_command = "CLOSE"
    else:
        gripper_command = "OPEN"
    
    print(f"\nGripper 控制:")
    print(f"  归一化值: {gripper_value:.4f}")
    print(f"  命令: {gripper_command}")
    print(f"  ⚠️  警告: 使用归一化值判断可能不正确!")
else:
    print(f"\n⚠️  错误: 无法提取gripper值!")

## 8. 可视化结果

### ❌ 问题 6: 可视化的值不是真实角度

In [ ]:
# ❌ 问题: 绘制的是归一化值，不是实际角度
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 绘制关节动作
joint_names = ['J0', 'J1', 'J2', 'J3', 'J4', 'J5']
ax1.bar(joint_names, predicted_action[:6], color='skyblue', edgecolor='black')
ax1.set_xlabel('关节')
ax1.set_ylabel('预测值 (归一化)')  # ❌ 应该是实际角度
ax1.set_title('Parol6 关节预测 (归一化值)')
ax1.grid(axis='y', alpha=0.3)

# 绘制gripper
if predicted_action.shape[0] >= 7:
    ax2.bar(['Gripper'], [predicted_action[6]], 
            color='lightcoral', edgecolor='black')
    ax2.set_ylabel('预测值 (归一化)')
    ax2.set_title('Gripper 预测 (归一化值)')
    ax2.axhline(y=0.5, color='r', linestyle='--', label='阈值=0.5 (错误!)')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('parol6_gripper_test_wrong.png', dpi=150, bbox_inches='tight')
print("\n⚠️  可视化已保存: parol6_gripper_test_wrong.png")
print("⚠️  注意: 图中显示的是归一化值，不是实际角度!")
plt.show()

## 总结问题列表

### ❌ 本版本存在的问题:

1. **缺少数据集加载**: 没有从数据集获取归一化统计信息
2. **Gripper维度错误**: `observation.state` 应该是7维 (6关节 + 1gripper)
3. **缺少反归一化**: 模型输出是归一化值，需要反归一化为实际角度
4. **Gripper阈值错误**: 不应该用归一化值直接判断gripper状态
5. **可视化误导**: 显示的是归一化值而不是实际角度
6. **缺少验证**: 没有验证输出值域是否合理

### ✅ 请查看 `9.test_parol6_gripper_fixed.ipynb` 获取修复版本!